# MDGR significance test on Kaggle (2× T4)

This notebook tests whether **MDGR** improves on the WaterSplatting **baseline** by more than run-to-run noise. Code: [noUpKey/test_water](https://github.com/noUpKey/test_water).

It trains both methods on the 4 SeaThru-NeRF scenes with seeds 43, 44 and 45: 24 runs in total, one per GPU at a time. It then runs a paired test.

**Design:**
- **Pairs:** a pair is `mdgr` and `baseline` on the same scene, with the same seed, in this session. The baseline PSNR shifted by up to 2 dB between Kaggle sessions, while same-session paired differences stayed within about 0.3 dB.
- **Scheduling:** jobs run pair by pair, so if the time budget runs out, the finished runs still form complete pairs.
- **Decision rule, fixed in advance:** the primary endpoint is the mean paired PSNR difference over all (scene, seed) pairs. MDGR counts as better only if:
    - the exact sign-flip permutation test gives p < 0.05; and
    - the 95 % confidence interval excludes 0.

  SSIM and LPIPS are secondary.
- **Pooling with the earlier run:** seeds 43–45 don't overlap the earlier seed-42 run, so its 4 pairs can be pooled afterwards: `python scripts/significance.py --roots <these outputs> <seed-42 outputs>`. Each pair still comes from a single session.

**Before running:**
1. *Settings → Accelerator → **GPU T4 ×2***, and *Settings → **Internet on***.
2. No dataset needed: the official SeaThru-NeRF zip (594 MB) is downloaded automatically.
3. Run with **Save Version → Save & Run All (Commit)**. The run takes longer than an interactive session lasts, and a committed version keeps `/kaggle/working` as downloadable output.

**Time:** about 45 min for the environment, then about 8.6 h of training, so about 9.5–10 h in total.

**Output** (`/kaggle/working/download/`):
- `pairs/pair_<scene>_seed<k>.zip`: written as soon as a pair finishes. Each holds both runs (eval, config, renders, debug strips, logs) plus weights-only checkpoints. These work with `ns-viewer`, `ns-render` and `ns-eval`, but can't resume training.
- `results_mdgr_significance.zip`: written at the end. It has the verdict, tables, logs and all runs, without checkpoints.

Raw runs stay on `/tmp`, so the saved output is a few dozen files. Kaggle can fail to save a version's output when it contains too many files.

**If training stops:**

| What happened | What the notebook does | What you do |
|---|---|---|
| One job crashes (e.g. out of memory) | Retries it once. If it fails again, that pair is left out of the test and listed as missing. | Nothing |
| Time budget reached | Starts no new jobs. Finished pairs are already saved. | Resume in a new version |
| The session dies (Kaggle error, 12 h limit, browser/kernel stop) | Every pair finished before the stop is already zipped in `download/pairs/`. A pair that was half done is lost. | Resume in a new version (below) |
| Kernel restart within an interactive session | `/tmp` and `/kaggle/working` survive, so finished runs are skipped. | *Run All* again |

**To resume:** create a new version of this notebook and attach the stopped version's output (*Add Input → Your Work → this notebook*). Section 7a imports every complete pair. Missing and half-finished pairs are trained from scratch, so both halves of every pair still come from the same session. The environment build (about 45 min) repeats.

**Saving interim results:** if the run is interactive, download each pair zip as it appears.

## 1. Configuration

In [ ]:
import os, sys, json, time, shutil, subprocess, threading, queue, glob, re, collections
from pathlib import Path

REPO_URL = "https://github.com/noUpKey/test_water.git"
REPO_REF = "main"

SCENES   = ["JapaneseGradens-RedSea", "IUI3-RedSea", "Curasao", "Panama"]
SEEDS    = [43, 44, 45]          # 3 seeds x 4 scenes = 12 (mdgr, baseline) pairs, all trained in this session
VARIANTS = ["mdgr", "baseline"]  # fixed: this notebook only tests MDGR against the baseline
GPUS     = [0, 1]

TIME_BUDGET_H    = 11.0   # do not START a job that would end after this (leaves time for packaging)
SKIP_DONE        = True   # skip runs whose eval.json already exists
DATA_ROOT        = ""     # "" = auto-discover under /kaggle/input
DATA_URL         = "https://drive.google.com/uc?export=download&id=1RzojBFvBWjUUhuJb95xJPSNP3nJwZWaT"
                           # official SeaThru-NeRF release (594 MB zip, sea-thru-nerf.github.io); used only if no dataset is attached
EXTRA_TRAIN_ARGS = []     # appended to every ns-train call, e.g. ["--max-num-iterations", "15000"]
CKPT_MODE        = "slim" # after evaluation: "slim" = keep weights only (~1/3 size; viewer, render and eval still work,
                          # resuming training does not), "full" = keep as is, "none" = delete checkpoints
RESUME_FROM      = "auto" # import COMPLETE pairs from an earlier version's output attached as input: "auto" = search
                          # /kaggle/input, "" = off, or a path. Half-finished pairs are retrained in full.
MAX_ATTEMPTS     = 2      # a failed job is retried once before its pair is given up
DELETE_CKPT_AFTER_ZIP = True  # remove a pair's checkpoints from /tmp once its zip is written and verified

WORK        = Path("/kaggle/working/ws")
SCRATCH     = Path("/tmp/ws_scratch")
ENV         = Path("/opt/ws_env")
MAMBA       = Path("/opt/micromamba")
COLMAP_ENV  = Path("/opt/colmap_env")
REPO        = SCRATCH / "src" / "water-splatting"
DATA_UNDIST = SCRATCH / "data_undistorted"
OUT         = SCRATCH / "outputs"   # raw runs live on /tmp (not saved); each finished pair is zipped to PAIRS
DL          = Path("/kaggle/working/download")
PAIRS       = DL / "pairs"          # one zip per completed (scene, seed) pair: both runs + checkpoints + logs
LOGS        = WORK / "logs"
RESULTS     = WORK / "results"
DIFIX_DIR   = SCRATCH / "third_party" / "Difix3D"
HF_HOME     = SCRATCH / "hf_cache"
for p in (WORK, SCRATCH, OUT, LOGS, RESULTS, HF_HOME, DATA_UNDIST, PAIRS):
    p.mkdir(parents=True, exist_ok=True)

TCNN_COMMIT = "daf9628c8d4500aee29935e15cff58a359277ad0"
T0 = time.time()

P = "--pipeline.model."
VARIANT_SPECS = {   # name -> (nerfstudio method, extra model args)
    "baseline":  ("water-splatting", []),
    "mdgr":      ("water-splatting-pseudo", [P + "pseudo-fixer", "difix_ref", P + "pseudo-fix-space", "clear"]),
}
assert all(v in VARIANT_SPECS for v in VARIANTS), "unknown variant"
NEEDS_DIFIX = any("difix" in " ".join(VARIANT_SPECS[v][1]) for v in VARIANTS)

PAPER = {  # Table 1 of the paper (PSNR, SSIM, LPIPS)
    "IUI3-RedSea": (29.386, 0.910, 0.180), "Curasao": (32.672, 0.957, 0.110),
    "JapaneseGradens-RedSea": (25.201, 0.904, 0.113), "Panama": (31.490, 0.948, 0.075),
}
print("variants:", VARIANTS, "| scenes:", SCENES, "| seeds:", SEEDS, "| difix needed:", NEEDS_DIFIX)

## 2. Helpers and GPU check

In [ ]:
def sh(cmd, log=None, env=None, cwd=None, check=True, echo=False, tail=30):
    """Runs a bash command. Full output goes to `log`; the last lines are printed on failure (or always if echo)."""
    log = Path(log) if log else LOGS / "misc.log"
    last = collections.deque(maxlen=tail)
    with open(log, "a") as f:
        f.write(f"\n$ {cmd}\n"); f.flush()
        proc = subprocess.Popen(["bash", "-c", cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, env=env, cwd=cwd, bufsize=1)
        for line in proc.stdout:
            f.write(line); last.append(line)
            if echo:
                print(line, end="")
        rc = proc.wait()
    if rc != 0 and check:
        print("".join(last))
        raise RuntimeError(f"command failed (exit {rc}), see {log}:\n{cmd}")
    return rc

def gpu_info():
    out = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total,compute_cap", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip().splitlines()
    return [tuple(x.strip() for x in l.split(",")) for l in out]

gpus = gpu_info()
for g in gpus:
    print("GPU", g)
assert gpus, "No GPU found: enable 'GPU T4 x2' in the notebook settings."
if len(gpus) < len(GPUS):
    print(f"WARNING: only {len(gpus)} GPU(s) visible, running jobs sequentially.")
    GPUS = [int(g[0]) for g in gpus]
CC = gpus[0][3]                       # e.g. "7.5" on T4
ARCH = CC.replace(".", "")

def env_vars(gpu=None, offline=False):
    e = dict(os.environ)
    for k in ("PYTHONPATH", "PYTHONHOME", "VIRTUAL_ENV", "CONDA_PREFIX", "MPLBACKEND"):
        e.pop(k, None)               # isolate the py3.8 env from the Kaggle kernel's python
    e.update({
        "PATH": f"{ENV}/bin:/usr/local/nvidia/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin",
        "CUDA_HOME": str(ENV),
        "LIBRARY_PATH": f"{ENV}/lib/stubs:{ENV}/lib:/usr/local/cuda/lib64/stubs",
        "CPATH": f"{ENV}/targets/x86_64-linux/include",
        "TCNN_CUDA_ARCHITECTURES": ARCH, "TORCH_CUDA_ARCH_LIST": CC, "MAX_JOBS": "4",
        "MAMBA_ROOT_PREFIX": str(MAMBA), "PYTHONNOUSERSITE": "1",
        "HF_HOME": str(HF_HOME), "DIFIX_SRC": str(DIFIX_DIR / "src"),
        "QT_QPA_PLATFORM": "offscreen", "OMP_NUM_THREADS": "2", "MKL_NUM_THREADS": "2",
        "PYTORCH_CUDA_ALLOC_CONF": "max_split_size_mb:256",   # less fragmentation when Difix and the Gaussians share a GPU
    })
    if gpu is not None:
        e["CUDA_VISIBLE_DEVICES"] = str(gpu)
    if offline:
        e["HF_HUB_OFFLINE"] = "1"; e["TRANSFORMERS_OFFLINE"] = "1"
    return e

PY = str(ENV / "bin" / "python")
def hours():
    return (time.time() - T0) / 3600
print("compute capability:", CC)

## 3. Build the environment (≈35–45 min the first time in a session)
This uses the same recipe as the reproduction: micromamba Python 3.8, torch 2.1.2+cu118, CUDA toolkit 11.8, tiny-cuda-nn at a pinned commit, and nerfstudio 1.1.4 with every package pinned from `reproduction/env/pip_freeze.txt`. `/opt` is not persisted between Kaggle sessions, so this cell rebuilds whatever is missing.

In [ ]:
LOG = LOGS / "build_env.log"
t = time.time()

def env_python_ok(stmt):
    return subprocess.run([PY, "-c", stmt], env=env_vars(), capture_output=True, cwd="/").returncode == 0

# --- micromamba + base env --------------------------------------------------------------------------------------
if not (MAMBA / "bin" / "micromamba").exists():
    sh(f"mkdir -p {MAMBA} && curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C {MAMBA} bin/micromamba", LOG)
if not Path(PY).exists():
    sh(f"{MAMBA}/bin/micromamba create -y -p {ENV} -c conda-forge python=3.8 gcc=11 gxx=11 ninja git", LOG, env_vars())
    sh(f"{PY} -m pip install pip==24.0 setuptools==69.5.1 wheel==0.44.0", LOG, env_vars())
print(f"[{time.time()-t:5.0f}s] base env ready")

# --- torch + CUDA toolkit ---------------------------------------------------------------------------------------
if not env_python_ok("import torch,sys; sys.exit(0 if torch.__version__.startswith('2.1.2') else 1)"):
    sh(f"{PY} -m pip install torch==2.1.2+cu118 torchvision==0.16.2+cu118 --extra-index-url https://download.pytorch.org/whl/cu118", LOG, env_vars())
if not (ENV / "bin" / "nvcc").exists():
    sh(f"{MAMBA}/bin/micromamba install -y -p {ENV} -c nvidia/label/cuda-11.8.0 -c conda-forge --strict-channel-priority cuda-toolkit", LOG, env_vars())
print(f"[{time.time()-t:5.0f}s] torch 2.1.2 + nvcc 11.8 ready")

# --- tiny-cuda-nn (slowest step) --------------------------------------------------------------------------------
if not env_python_ok("import tinycudann"):
    print("compiling tiny-cuda-nn (~15-25 min) ...")
    sh(f"{PY} -m pip install --no-build-isolation git+https://github.com/NVlabs/tiny-cuda-nn/@{TCNN_COMMIT}#subdirectory=bindings/torch", LOG, env_vars())
print(f"[{time.time()-t:5.0f}s] tiny-cuda-nn ready")

# --- repository ---------------------------------------------------------------------------------------------------
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    if tok:
        url = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass
if REPO.exists():
    shutil.rmtree(REPO)
sh(f"git clone --recursive {url} {REPO} && cd {REPO} && git checkout {REPO_REF} && git submodule update --init --recursive", LOG, env_vars())
commit = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("repo commit:", commit)

# --- pinned python deps (nerfstudio 1.1.4 stack) -----------------------------------------------------------------
CONSTRAINTS = SCRATCH / "constraints_ns114_py38.txt"
skip = ("torch==", "torchvision==", "tinycudann", "water-splatting", "water_splatting")
FREEZE = REPO / "reproduction/env/pip_freeze.txt"
assert FREEZE.exists(), (f"{FREEZE} is missing from the cloned repo ({commit[:8]}). Check REPO_URL / REPO_REF "
                         "point to a commit that contains reproduction/env/pip_freeze.txt.")
lines = [l for l in FREEZE.read_text().splitlines()
         if l.strip() and " @ " not in l and not l.startswith(("-e", "#")) and not l.startswith(skip)]
CONSTRAINTS.write_text("\n".join(lines) + "\n")
if not env_python_ok("import sys; from importlib.metadata import version; sys.exit(0 if version('nerfstudio') == '1.1.4' else 1)"):
    sh(f"{PY} -m pip install -r {CONSTRAINTS}", LOG, env_vars())
assert env_python_ok("import torch,sys; sys.exit(0 if torch.__version__.startswith('2.1.2') else 1)"), "torch was replaced by a dependency!"
print(f"[{time.time()-t:5.0f}s] nerfstudio stack ready")

# --- WaterSplatting (always rebuilt so the CUDA kernels match the checked-out code) -------------------------------
sh("rm -rf ~/.cache/torch_extensions/*/water_splatting_cuda", LOG, check=False)
sh(f"cd {REPO} && {PY} -m pip install --no-deps --force-reinstall --no-use-pep517 --no-build-isolation .", LOG, env_vars())
# require the compiled extension: the package would otherwise silently fall back to a JIT build
assert env_python_ok("from water_splatting import csrc; import tinycudann, water_splatting.cuda"), f"CUDA extension missing, see {LOG}"
sh(f"{ENV}/bin/ns-train water-splatting-pseudo --help > /dev/null", LOG, env_vars(), cwd="/")
print(f"[{time.time()-t:5.0f}s] water-splatting {commit[:8]} installed, methods water-splatting / water-splatting-pseudo registered")

## 4. Difix fixer (only if an MDGR / Difix variant is selected)
Difix pins `diffusers 0.25.1 / transformers 4.38 / peft 0.9`, which still support Python 3.8, so it shares the env. The weights are downloaded once, here, and the training jobs then run offline, so the two GPUs don't race on the HuggingFace cache. A short smoke test runs the fixer on GPU 0 before any hours are spent. *Difix is released under the NVIDIA license (non-commercial research use).*

In [ ]:
if NEEDS_DIFIX:
    LOG = LOGS / "difix_setup.log"
    if not (DIFIX_DIR / "src" / "pipeline_difix.py").exists():
        sh(f"rm -rf {DIFIX_DIR} && git clone --depth 1 https://github.com/nv-tlabs/Difix3D.git {DIFIX_DIR}", LOG, env_vars())
    sh(f'{PY} -m pip install "diffusers==0.25.1" "transformers==4.38.0" "peft==0.9.0" "huggingface-hub==0.25.1" '
       f'"accelerate==0.27.2" einops', LOG, env_vars())
    assert subprocess.run([PY, "-c", "import torch,sys; sys.exit(0 if torch.__version__.startswith('2.1.2') else 1)"],
                          env=env_vars()).returncode == 0, "torch was replaced while installing Difix deps!"
    sh(f"""{PY} -c "from huggingface_hub import snapshot_download as s; print(s('nvidia/difix_ref'))" """, LOG, env_vars())
    smoke = SCRATCH / "difix_smoke.py"
    smoke.write_text(
        "import sys, os, time, torch\n"
        "sys.path.insert(0, os.environ['DIFIX_SRC'])\n"
        "from water_splatting.pseudo_views import DifixFixer\n"
        "f = DifixFixer('nvidia/difix_ref', use_reference=True, fp16=True)\n"
        "img = torch.rand(600, 900, 3); ref = torch.rand(600, 900, 3)\n"
        "t = time.time(); out = f([img], [ref])[0]\n"
        "assert out.shape == img.shape and torch.isfinite(out).all(), out.shape\n"
        "print(f'Difix OK: {tuple(out.shape)} in {time.time()-t:.1f}s, mean {out.mean():.3f}')\n")
    sh(f"{PY} {smoke}", LOG, env_vars(gpu=GPUS[0], offline=True), cwd=str(SCRATCH), echo=True)
else:
    print("no Difix variant selected, skipping")

## 5. Data: discover SeaThru-NeRF, undistort with COLMAP
This follows the README protocol: `colmap image_undistorter` on `images_wb` plus the provided COLMAP model. The test split is **every 8th image in alphabetical order** (`--eval-mode interval --eval-interval 8`), the same as SeaThru-NeRF and the paper.

In [ ]:
LOG = LOGS / "data.log"

def norm_key(name):
    """Name normalisation for matching scene folders: case / separators / known spelling variants."""
    k = re.sub(r"[^a-z0-9]", "", name.lower())
    return k.replace("gardens", "gradens").replace("curacao", "curasao")

def scene_matches(dirname, scene):
    d, s = norm_key(dirname), norm_key(scene)
    return d == s or d == s.replace("redsea", "") or d.replace("redsea", "") == s.replace("redsea", "")

def find_dirs(root, scene, max_depth=9):
    root = Path(root); hits = []
    for dirpath, dirnames, _ in os.walk(root, followlinks=True):
        if len(Path(dirpath).relative_to(root).parts) >= max_depth:
            dirnames[:] = []; continue
        for d in dirnames:
            if scene_matches(d, scene):
                hits.append(Path(dirpath) / d)
    return sorted(hits, key=lambda p: len(p.parts))

def subdir_ci(path, name):
    for c in Path(path).iterdir():
        if c.is_dir() and c.name.lower() == name.lower():
            return c
    return None

def classify(scene_dir):
    """('undistorted', images, sparse) | ('raw', images_wb, sparse_model) | None"""
    imgs, sparse = subdir_ci(scene_dir, "images"), subdir_ci(scene_dir, "sparse")
    if imgs and sparse and ((sparse / "cameras.bin").exists() or (sparse / "cameras.txt").exists()):
        return ("undistorted", imgs, sparse)
    wb = subdir_ci(scene_dir, "images_wb")
    models = sorted({p.parent for p in list(Path(scene_dir).rglob("cameras.bin")) + list(Path(scene_dir).rglob("cameras.txt"))},
                    key=lambda p: (0 if p.name == "0" else 1, len(p.parts)))
    if wb and models:
        return ("raw", wb, models[0])
    return None

def roots():
    if DATA_ROOT:
        return [Path(DATA_ROOT)]
    return [DATA_UNDIST, Path("/kaggle/input"), SCRATCH / "raw"]   # DATA_UNDIST: reuse earlier undistortion

def discover(scene):
    for r in roots():
        if r.exists():
            for d in find_dirs(r, scene):
                c = classify(d)
                if c:
                    return d, c
    return None, None

def extract_archives(src_root, dst_root, max_depth=5):
    """Kaggle usually unpacks uploaded archives, but not archives nested inside a dataset."""
    n = 0
    for dirpath, dirnames, files in os.walk(src_root):
        if len(Path(dirpath).relative_to(src_root).parts) >= max_depth:
            dirnames[:] = []
        for fn in files:
            if fn.lower().endswith((".zip", ".tar", ".tar.gz", ".tgz", ".7z")):
                out = dst_root / re.sub(r"\.(zip|tar|tar\.gz|tgz|7z)$", "", fn, flags=re.I)
                if out.exists():
                    continue
                out.mkdir(parents=True)
                arc = Path(dirpath) / fn
                print("extracting", arc)
                cmd = (f"unzip -q -o '{arc}' -d '{out}'" if fn.lower().endswith(".zip") else
                       f"7z x -y -o'{out}' '{arc}'" if fn.lower().endswith(".7z") else f"tar -xf '{arc}' -C '{out}'")
                sh(cmd, LOG, check=False); n += 1
    return n

def show_tree(root, max_depth=4, max_lines=120):
    root = Path(root); lines = 0
    for dirpath, dirnames, files in os.walk(root):
        rel = Path(dirpath).relative_to(root)
        if len(rel.parts) > max_depth:
            dirnames[:] = []; continue
        dirnames.sort()
        print("  " * len(rel.parts) + f"{Path(dirpath).name}/  ({len(files)} files)")
        lines += 1
        if lines >= max_lines:
            print("  ..."); return

raw = SCRATCH / "raw"; raw.mkdir(exist_ok=True)
if not all(discover(s)[0] for s in SCENES) and Path("/kaggle/input").exists():
    extract_archives(Path("/kaggle/input"), raw)
if DATA_URL and not all(discover(s)[0] for s in SCENES):
    if "drive.google.com" in DATA_URL:
        sh(f"pip install -q gdown && cd {raw} && (gdown --fuzzy --folder '{DATA_URL}' || gdown --fuzzy '{DATA_URL}')", LOG)
    else:
        sh(f"cd {raw} && wget -q -O data_archive '{DATA_URL}'", LOG)
    sh(f"cd {raw} && for f in *; do case \"$f\" in *.zip) unzip -q -o \"$f\";; *.tar*|*.tgz) tar -xf \"$f\";; esac; done; "
       f"[ -f data_archive ] && (unzip -q -o data_archive || tar -xf data_archive) || true", LOG, check=False)

missing = [s for s in SCENES if discover(s)[0] is None]
if missing:
    print("Could not find:", missing, "\nThis is what is mounted under /kaggle/input:")
    show_tree("/kaggle/input")
    raise FileNotFoundError(
        f"scenes {missing} not found. Attach the SeaThru-NeRF dataset (Add Input), or set DATA_ROOT to the folder that "
        "contains the scene folders. Each scene needs images_wb/ + a COLMAP model (cameras.bin), or images/ + sparse/cameras.bin.")

need_colmap = False
plan = {}
for s in SCENES:
    d, c = discover(s)
    plan[s] = c
    need_colmap |= c[0] == "raw" and not (DATA_UNDIST / s / "sparse" / "cameras.bin").exists()
    print(f"{s:24s} {c[0]:12s} images={c[1]}  model={c[2]}")

if need_colmap and not (COLMAP_ENV / "bin" / "colmap").exists():
    sh(f"{MAMBA}/bin/micromamba create -y -p {COLMAP_ENV} -c conda-forge colmap", LOGS / "colmap_install.log", env_vars())

DATA_UNDIST.mkdir(parents=True, exist_ok=True)   # colmap does not create parent directories
SPLITS = {}
for s, (kind, imgs, model) in plan.items():
    dst = DATA_UNDIST / s
    if kind == "undistorted":
        if imgs.parent.resolve() == dst.resolve():
            pass                              # already prepared in this session
        elif not (dst / "sparse").exists():   # copy: /kaggle/input is read-only
            shutil.rmtree(dst, ignore_errors=True)
            dst.mkdir(parents=True)
            shutil.copytree(imgs, dst / "images")
            shutil.copytree(model, dst / "sparse")
    elif not (dst / "sparse" / "cameras.bin").exists():
        shutil.rmtree(dst, ignore_errors=True)
        sh(f"{COLMAP_ENV}/bin/colmap image_undistorter --image_path '{imgs}' --input_path '{model}' "
           f"--output_path '{dst}' --output_type COLMAP", LOGS / f"undistort_{s}.log", env_vars())
    names = sorted(p.name for p in (dst / "images").iterdir() if p.suffix.lower() in (".png", ".jpg", ".jpeg"))
    test = [n for i, n in enumerate(names) if i % 8 == 0]
    SPLITS[s] = {"n_total": len(names), "n_train": len(names) - len(test), "n_test": len(test), "test": test}
    print(f"{s:24s} {len(names)} images -> train {len(names)-len(test)} / test {len(test)}  {test}")
(RESULTS / "splits.json").write_text(json.dumps(SPLITS, indent=2))

## 6. Sanity checks (unit tests and CUDA depth-gradient check)

In [ ]:
LOG = LOGS / "sanity.log"
sh(f"{PY} {REPO}/tests/test_pseudo_views.py", LOG, env_vars(), cwd=str(WORK), echo=True)
sh(f"{PY} {REPO}/tests/test_significance.py", LOG, env_vars(), cwd=str(WORK), echo=True)
# run from WORK (not the repo) so `import water_splatting` resolves to the installed, compiled package
sh(f"{PY} {REPO}/scripts/check_depth_grad.py", LOG, env_vars(gpu=GPUS[0]), cwd=str(WORK), echo=True)

## 7a. Pair saving and resume
Each completed pair is zipped to `/kaggle/working/download/pairs/` as soon as its second run finishes. A pair zip holds both runs (eval, config, renders, debug strips, checkpoints) and their logs. Raw runs stay on `/tmp`, so the saved output is a few dozen files rather than thousands.

**To resume after a stop:**
1. Open the stopped version's page and attach its output to a new version: *Add Input → Your Work → that notebook*.
2. Run the new version.

This cell imports every complete pair it finds under `/kaggle/input`. Pairs that were half-finished, or missing, are trained again from scratch. Each pair therefore always comes from a single session.

In [ ]:
import zipfile

def run_dir_of(scene, variant, seed):
    return OUT / scene / VARIANT_SPECS[variant][0] / f"{variant}_seed{seed}"

def pair_zip(scene, seed):
    return PAIRS / f"pair_{scene}_seed{seed}.zip"

def pair_done(scene, seed):
    return all((run_dir_of(scene, v, seed) / "eval.json").exists() for v in VARIANTS)

def zip_pair(scene, seed):
    # Writes the pair zip once both runs are evaluated; verified before checkpoints are deleted. Returns the zip or None.
    zp = pair_zip(scene, seed)
    if zp.exists() or not pair_done(scene, seed):
        return zp if zp.exists() else None
    part = zp.with_name(zp.name + ".part")
    with zipfile.ZipFile(part, "w", zipfile.ZIP_STORED, allowZip64=True) as z:   # .ckpt / .png barely compress
        for v in VARIANTS:
            for d in (run_dir_of(scene, v, seed), OUT / scene / f"mdgr_debug_{v}_seed{seed}"):
                for f in sorted(d.rglob("*")) if d.exists() else []:
                    if f.is_file() and f.suffix != ".tmp":
                        z.write(f, Path("outputs") / f.relative_to(OUT))
            log = LOGS / f"{scene}_{v}_seed{seed}.log"
            if log.exists():
                z.write(log, f"logs/{log.name}")
    with zipfile.ZipFile(part) as z:
        bad = z.testzip()
    if bad is not None:
        part.unlink()
        raise RuntimeError(f"corrupt entry {bad} while zipping pair {scene} seed {seed}; checkpoints kept")
    part.replace(zp)
    if DELETE_CKPT_AFTER_ZIP:
        for v in VARIANTS:
            for c in (run_dir_of(scene, v, seed) / "nerfstudio_models").glob("*.ckpt"):
                c.unlink()
    return zp

def import_pair(src):
    # Imports a complete pair from an earlier version (a pair zip, or the folder Kaggle unpacked it into).
    m = re.fullmatch(r"pair_(.+)_seed(\d+)(\.zip)?", src.name)
    if not m or (m.group(1), int(m.group(2))) in imported:
        return None
    scene, seed = m.group(1), int(m.group(2))
    if scene not in SCENES or seed not in SEEDS or pair_zip(scene, seed).exists():
        return None
    need = [f"outputs/{scene}/{VARIANT_SPECS[v][0]}/{v}_seed{seed}/eval.json" for v in VARIANTS]
    if src.is_file() and src.suffix == ".zip":
        with zipfile.ZipFile(src) as z:
            names = set(z.namelist())
            if not all(n in names for n in need):
                return None
            z.extractall(SCRATCH, members=[n for n in names if n.startswith("outputs/") and not n.endswith(".ckpt")])
        shutil.copy(src, pair_zip(scene, seed))
    else:
        if not all((src / n).exists() for n in need):
            return None
        shutil.copytree(src / "outputs", OUT, dirs_exist_ok=True,
                        ignore=lambda d, names: [n for n in names if n.endswith(".ckpt")])
        shutil.make_archive(str(pair_zip(scene, seed))[:-4], "zip", src)
    return scene, seed

imported = set()
if RESUME_FROM:
    base = Path("/kaggle/input") if RESUME_FROM == "auto" else Path(RESUME_FROM)
    for src in sorted(base.rglob("pair_*_seed*")) if base.exists() else []:
        if src.suffix == ".zip" or src.is_dir():
            got = import_pair(src)
            if got:
                imported.add(got)
                print(f"resumed pair {got[0]} seed {got[1]} from {src}")
# pairs finished earlier in THIS session (e.g. after a kernel restart) but not yet zipped
for seed in SEEDS:
    for s in SCENES:
        zip_pair(s, seed)
print(f"{len(imported)} pairs imported, {len(list(PAIRS.glob('pair_*.zip')))} of {len(SEEDS) * len(SCENES)} pairs saved")

## 7. Train + evaluate on both GPUs
Each GPU has a worker that pulls `(scene, variant, seed)` jobs from a shared queue. Jobs are queued pair by pair, so the two GPUs usually train both halves of a pair at the same time. Each job runs `ns-train`, then `ns-eval` with renders. Per-job logs go to `/kaggle/working/ws/logs/`. A status table is printed every 10 minutes. Jobs that would overrun `TIME_BUDGET_H` are not started.

In [ ]:
# --- guard: a previous execution of this cell keeps running after an interrupt (its worker threads live in the
# kernel and its ns-train processes are not killed), which would put two jobs on one GPU -> CUDA out of memory.
if "STOP" in globals():
    STOP.set()                                   # old workers exit after their current job
subprocess.run(["pkill", "-f", f"{ENV}/bin/ns-(train|eval)"], check=False)
time.sleep(10)
STOP = threading.Event()
used = subprocess.run(["nvidia-smi", "--query-gpu=index,memory.used", "--format=csv,noheader,nounits"],
                      capture_output=True, text=True).stdout.split()
print("GPU memory in use (MiB) before start:", used)
busy = [used[i].strip(",") for i in range(0, len(used), 2) if int(used[i + 1].strip(",")) > 1000]
assert not busy, (f"GPU(s) {busy} still hold memory from an earlier run: restart the kernel and run all cells again "
                  "(finished runs are skipped).")

DATA_ARGS = ["colmap", "--downscale-factor", "1", "--colmap-path", "sparse", "--images-path", "images",
             "--eval-mode", "interval", "--eval-interval", "8"]

def run_dir_of(scene, variant, seed):
    return OUT / scene / VARIANT_SPECS[variant][0] / f"{variant}_seed{seed}"

def run_job(gpu, scene, variant, seed):
    method, extra = VARIANT_SPECS[variant]
    ts = f"{variant}_seed{seed}"
    rd = run_dir_of(scene, variant, seed)
    log = LOGS / f"{scene}_{ts}.log"
    if rd.exists():
        shutil.rmtree(rd)                     # unfinished earlier attempt
    extra = list(extra)
    if method == "water-splatting-pseudo":
        extra += [P + "pseudo-debug-dir", str(OUT / scene / f"mdgr_debug_{ts}")]
    train = [f"{ENV}/bin/ns-train", method, "--vis", "tensorboard", "--output-dir", str(OUT),
             "--experiment-name", scene, "--timestamp", ts, "--machine.seed", str(seed),
             *extra, *EXTRA_TRAIN_ARGS, *DATA_ARGS, "--data", str(DATA_UNDIST / scene)]
    env = env_vars(gpu=gpu, offline=True)
    t = time.time()
    sh(" ".join(f"'{a}'" for a in train), log, env, cwd=str(WORK))
    train_min = (time.time() - t) / 60
    sh(f"{ENV}/bin/ns-eval --load-config '{rd}/config.yml' --output-path '{rd}/eval.json' "
       f"--render-output-path '{rd}/renders'", log, env, cwd=str(WORK))
    meta = {"scene": scene, "variant": variant, "method": method, "seed": seed, "gpu": gpu,
            "train_minutes": round(train_min, 2), "total_minutes": round((time.time() - t) / 60, 2),
            "gpu_name": gpus[0][1], "repo_commit": commit, "command": " ".join(train)}
    (rd / "run_meta.json").write_text(json.dumps(meta, indent=2))
    if CKPT_MODE == "none":
        shutil.rmtree(rd / "nerfstudio_models", ignore_errors=True)
    elif CKPT_MODE == "slim":
        slim_checkpoints(rd)
    return train_min

def slim_checkpoints(rd):
    """Drops optimizer / scheduler state: eval_setup (viewer, ns-render, ns-eval) loads only `pipeline` and `step`."""
    import torch
    for c in (rd / "nerfstudio_models").glob("*.ckpt"):
        state = torch.load(c, map_location="cpu", weights_only=False)
        tmp = c.with_suffix(".tmp")
        torch.save({"step": state["step"], "pipeline": state["pipeline"]}, tmp)
        del state
        tmp.replace(c)

jobs = queue.Queue()
status = collections.OrderedDict()
for seed in SEEDS:          # pair by pair: if the time budget runs out, finished runs still form complete pairs
    for s in SCENES:
        for v in VARIANTS:
            key = (s, v, seed)
            if (s, seed) in imported:
                status[key] = "done (resumed from earlier version)"
            elif SKIP_DONE and (run_dir_of(s, v, seed) / "eval.json").exists():
                status[key] = "done (earlier in this session)"
            else:
                status[key] = "queued"; jobs.put(key)

durations = collections.defaultdict(list)          # variant -> minutes, to estimate remaining jobs
DEFAULT_MIN = {"baseline": 45, "depthgrad": 50}     # pseudo-view variants default to 60
lock = threading.Lock()
attempts = collections.Counter()

def last_error(log):
    try:
        lines = [l.strip() for l in open(log, errors="ignore") if l.strip()]
    except OSError:
        return ""
    errs = [l for l in lines if re.search(r"(Error|Exception|Killed)", l)]
    return (errs or lines or [""])[-1][:160]

def worker(gpu):
    while not STOP.is_set():
        try:
            key = jobs.get_nowait()
        except queue.Empty:
            return
        s, v, seed = key
        with lock:
            est = (sum(durations[v]) / len(durations[v])) if durations[v] else DEFAULT_MIN.get(v, 60)
        if hours() + est / 60 > TIME_BUDGET_H:
            status[key] = "skipped (time budget)"; continue
        status[key] = f"running on GPU {gpu} (since {hours():.2f} h)"
        t = time.time()
        attempts[key] += 1
        try:
            run_job(gpu, s, v, seed)
        except Exception as e:
            err = last_error(LOGS / f"{s}_{v}_seed{seed}.log") or str(e).splitlines()[0][:80]
            if attempts[key] < MAX_ATTEMPTS:
                status[key] = f"retrying after failure: {err}"; jobs.put(key)
            else:
                status[key] = f"FAILED after {attempts[key]} attempts: {err}"
            continue
        with lock:
            durations[v].append((time.time() - t) / 60)
            try:
                zp = zip_pair(s, seed)          # under the lock: both halves may finish at the same time
            except Exception as e:
                zp = None; print(f"!! could not zip pair {s} seed {seed}: {e}")
        status[key] = f"done in {(time.time() - t) / 60:.0f} min (GPU {gpu})" + (f", pair saved" if zp else "")

def print_status():
    print(f"--- {time.strftime('%H:%M:%S')}  elapsed {hours():.2f} h ---")
    for (s, v, seed), st in status.items():
        print(f"  {s:24s} {v:10s} seed {seed:<4d} {st}")

threads = [threading.Thread(target=worker, args=(g,), daemon=True) for g in GPUS]
for i, th in enumerate(threads):
    th.start()
    time.sleep(90 if i == 0 else 0)   # stagger start-up (image caching / CPU contention)
last = 0
while any(th.is_alive() for th in threads):
    time.sleep(30)
    if time.time() - last > 600:
        print_status(); last = time.time()
print_status()

## 8. Results

In [ ]:
import pandas as pd

rows = []
for ev in sorted(OUT.glob("*/*/*_seed*/eval.json")):
    rd = ev.parent
    scene = rd.parts[-3]
    variant, seed = rd.name.rsplit("_seed", 1)
    r = json.loads(ev.read_text())["results"]
    meta = json.loads((rd / "run_meta.json").read_text()) if (rd / "run_meta.json").exists() else {}
    rows.append({"scene": scene, "variant": variant, "seed": int(seed), "psnr": r["psnr"], "ssim": r["ssim"],
                 "lpips": r["lpips"], "train_min": meta.get("train_minutes")})
df = pd.DataFrame(rows)
assert len(df), "no finished runs found"
df.to_csv(RESULTS / "per_run.csv", index=False)
pd.set_option("display.width", 200)

wide = df.pivot_table(index=["scene", "seed"], columns="variant", values="psnr")
if {"mdgr", "baseline"} <= set(wide.columns):
    wide["diff"] = wide["mdgr"] - wide["baseline"]
print("PSNR per run (dB):\n", wide.round(3).to_string())

per_scene = df.groupby(["scene", "variant"])[["psnr", "ssim", "lpips", "train_min"]].agg(["mean", "std"])
per_scene.to_csv(RESULTS / "per_scene.csv")
print("\nPer scene, mean and SD over seeds:\n", per_scene.round(4).to_string())

overall = df.groupby(["variant", "seed"])[["psnr", "ssim", "lpips"]].mean().groupby("variant").agg(["mean", "std"])
overall.to_csv(RESULTS / "average.csv")
print("\nAverage over the 4 scenes (mean and SD over seeds):\n", overall.round(4).to_string())
print(f"Paper average PSNR (3 runs, RTX 4080): {sum(v[0] for v in PAPER.values()) / len(PAPER):.3f}")

### Significance test: MDGR vs baseline
This applies the decision rule from the top of the notebook. Besides the verdict, it reports the smallest effect this number of pairs can reliably detect (80 % power). A non-significant difference smaller than that is inconclusive, not evidence that MDGR has no effect.

In [ ]:
res = subprocess.run([PY, str(REPO / "scripts/significance.py"), "--a", "mdgr", "--b", "baseline",
                      "--out", str(RESULTS / "significance_mdgr_vs_baseline"), "--roots", str(OUT)],
                     capture_output=True, text=True, cwd=str(WORK))
print(res.stdout or res.stderr)

In [ ]:
# Region-wise PSNR (centre -> corner rings): MDGR targets the poorly observed image borders.
dirs = sorted(str(p) for p in OUT.glob("*/*/*_seed*/renders"))
res = subprocess.run([PY, str(REPO / "scripts/radial_metrics.py"), *dirs], capture_output=True, text=True,
                     env=env_vars(), cwd=str(WORK))
print(res.stdout or res.stderr)
(RESULTS / "radial_metrics.txt").write_text(res.stdout + res.stderr)

In [ ]:
# MDGR debug strips: [render | fixer input | fixer output | pseudo-target | reference]
from IPython.display import Image as IPImage, display
strips = sorted(OUT.glob("*/mdgr_debug_mdgr_seed*/*.jpg"))
print(len(strips), "debug strips")
for p in strips[-3:]:
    print(p.relative_to(OUT)); display(IPImage(filename=str(p), width=1400))

## 9. Package results for download
Everything is in `/kaggle/working/download/`:
- `results_mdgr_significance.zip`: the verdict, tables, logs, and per-run `eval.json`, configs, renders and debug strips. It has no checkpoints.
- `pairs/pair_<scene>_seed<k>.zip`: one per completed pair, written during training. Each holds both runs, including their checkpoints. These are weights only: they work with `ns-viewer`, `ns-render` and `ns-eval`, but can't resume training.

Re-running this cell is safe.

In [ ]:
from IPython.display import FileLink, display

def add_tree(z, root, arc_root, skip=lambda f: False):
    root = Path(root)
    if root.exists():
        for f in sorted(root.rglob("*")):
            if f.is_file() and not skip(f):
                z.write(f, Path(arc_root) / f.relative_to(root))

def free_gb():
    return shutil.disk_usage("/kaggle/working").free / 1e9

# 1) results: verdict + tables, logs, per-run eval.json / config.yml / run_meta.json / tensorboard, renders, debug strips
res_zip = DL / "results_mdgr_significance.zip"
with zipfile.ZipFile(res_zip, "w", zipfile.ZIP_DEFLATED, allowZip64=True) as z:
    add_tree(z, RESULTS, "results")
    add_tree(z, LOGS, "logs")
    add_tree(z, OUT, "outputs", skip=lambda f: f.suffix in (".ckpt", ".tmp"))
print(f"{res_zip.name}: {res_zip.stat().st_size / 1e6:.0f} MB | free disk {free_gb():.1f} GB")

# 2) pair zips (with checkpoints) are written during training; catch any complete pair not saved yet
for seed in SEEDS:
    for s in SCENES:
        zip_pair(s, seed)
for part in PAIRS.glob("*.part"):
    part.unlink()                                  # leftovers of an interrupted zip; the pair is re-zipped above
missing = [(s, seed) for seed in SEEDS for s in SCENES if not pair_zip(s, seed).exists()]
print(f"{len(SEEDS) * len(SCENES) - len(missing)} of {len(SEEDS) * len(SCENES)} pairs saved"
      + (f"; missing: {missing} (resume in a new version to train them)" if missing else ""))
print(f"files in /kaggle/working: {sum(1 for f in Path('/kaggle/working').rglob('*') if f.is_file())} | free disk {free_gb():.1f} GB")

os.chdir("/kaggle/working")
print("\nDownload (or, after a committed run, use the version's Output tab):")
for p in sorted(DL.rglob("*.zip")):
    print(f"  {str(p.relative_to(DL)):55s} {p.stat().st_size / 1e6:9.1f} MB")
    display(FileLink(str(p.relative_to("/kaggle/working"))))